# Project: Build Your Own AI Agent

# 📄 Introduction

Welcome to the project! In this assignment, you'll build a simple AI agent that can interpret user input and call helpful tools based on what it understands. You'll start by defining a set of tools, then implement an agent loop that reads natural language input, selects the appropriate tool, calls it, and returns a result (called the final answer).

The goal is **not to use any complex frameworks, but to understand the logic behind AI agent systems**. Optionally, you can use simple packages like LangChain or SmolAgent if you're confident, but a plain Python approach is encouraged.

---

## 🌐 Project Goal
Design an AI agent that:
- Accepts user input (a question or instruction)
- Identifies which tool (function) to use
- Extracts relevant arguments from the input
- Calls the function with those arguments
- Returns an answer to the user

---

## 📊 What You Will Do

1. Define at least 2 tools (functions) that your agent can use. Example tools:
   - get_weather(location)
   - calculator_add(a, b) ← adds two numbers
   - calculator_multiply(a, b) ← multiplies two numbers
   - generate_image(prompt)
   - get_current_time(city) ← returns current time in a city
   - search_wikipedia(topic) ← fetches a short summary from Wikipedia
   - get_distance_between(from_location, to_location) ← uses a maps API to get the distance between two places

   You are also encouraged to create your own tools if you have other ideas. Just make sure to explain what your tool does and why it would be useful for an LLM-based agent.

2. Simulate or integrate an LLM that returns an "Action" JSON block indicating what tool to call and with what input.

3. Parse that Action block and use the tool accordingly.

4. Print the "Final Answer" (result) to the user.

5. (Optional) Let the loop continue for another user question.

---

## ✨ Bonus Features (Optional)
- Publish your agent in the huggingface space via gradio.
- Add image/speech generation as a tool (calling another model from huggingface).

---

### Hugging Face

✅ 1. **Create a free Hugging Face account** at [https://huggingface.co/](https://huggingface.co/)

✅ 2. (Optional but recommended) **Create an Access Token**:
   - Go to [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
   - Click **New Token**
   - Name it (e.g., `colab`)
   - Set permission as **Read**
   - Copy the token (we'll use it if needed)

✅ 3. **You only need to use the token if you want to download larger models** (e.g., Falcon, GPT-Neo).
   - For small models like `distilgpt2`, `gpt2`, you don't need any token.
___
Start writing your code below this section. You may organize your notebook in sections like:
- Tool definitions
- LLM simulation
- Agent loop
- Input/output examples

---

📅 Deadline: 19/12/2025

📌 Submission Format:

Jupyter Notebook (.ipynb).

Conversation with AI Chatbot (if used, attach the conversation along with a brief report explaining what you learned from it)

TP should be done individually.

Good luck.

In [17]:
# =========================================
# Project: Build Your Own AI Agent
# Hugging Face LLM (Hybrid Version) + Gradio UI
# =========================================

import json
import re
from datetime import datetime
from zoneinfo import ZoneInfo
from transformers import pipeline
import gradio as gr

# =========================================
# LOAD HUGGING FACE MODEL
# =========================================

print("Loading Hugging Face model...")
llm = pipeline(
    "text-generation",
    model="distilgpt2",
    truncation=True,
    pad_token_id=50256
)
print("Model loaded successfully!")

# =========================================
# TOOL DEFINITIONS
# =========================================

def calculator_add(a: float, b: float) -> float:
    return a + b

def calculator_multiply(a: float, b: float) -> float:
    return a * b

def get_current_time(city: str) -> str:
    city_timezones = {
        "london": "Europe/London",
        "new york": "America/New_York",
        "paris": "Europe/Paris",
        "tokyo": "Asia/Tokyo",
        "delhi": "Asia/Kolkata",
        "dubai": "Asia/Dubai",
        "sydney": "Australia/Sydney"
    }

    city_lower = city.lower()
    if city_lower not in city_timezones:
        return f"Sorry, I don't have timezone data for {city}."

    tz = ZoneInfo(city_timezones[city_lower])
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")

    return f"The current time in {city.title()} is {now}"

# =========================================
# HUGGING FACE + LOGIC REASONING
# =========================================

def huggingface_reasoning(user_input: str) -> dict:
    """
    Hugging Face model understands intent,
    Python ensures correct action extraction.
    """

    # Use LLM for semantic understanding (soft reasoning)
    prompt = f"Understand this request: {user_input}"
    llm(prompt, max_new_tokens=20)

    text = user_input.lower()

    # Extract numbers
    numbers = re.findall(r"-?\d+\.?\d*", text)
    numbers = [float(n) for n in numbers]

    # ADD
    if "add" in text and len(numbers) >= 2:
        return {
            "action": "calculator_add",
            "arguments": {"a": numbers[0], "b": numbers[1]}
        }

    # MULTIPLY
    if "multiply" in text and len(numbers) >= 2:
        return {
            "action": "calculator_multiply",
            "arguments": {"a": numbers[0], "b": numbers[1]}
        }

    # TIME
    if "time" in text:
        match = re.search(r"time in ([a-zA-Z ]+)", text)
        city = match.group(1).strip() if match else "London"
        return {
            "action": "get_current_time",
            "arguments": {"city": city}
        }

    return {"action": "none", "arguments": {}}

# =========================================
# AGENT SETUP
# =========================================

TOOLS = {
    "calculator_add": calculator_add,
    "calculator_multiply": calculator_multiply,
    "get_current_time": get_current_time
}

def ai_agent(user_input: str) -> str:
    """
    Core agent logic, returns final answer as string
    """
    action_data = huggingface_reasoning(user_input)
    action = action_data["action"]
    arguments = action_data["arguments"]

    if action in TOOLS:
        result = TOOLS[action](**arguments)
        return f"Final Answer: {result}"
    else:
        return "Sorry, I don't understand your request."

# =========================================
# GRADIO UI SETUP
# =========================================

def launch_gradio():
    with gr.Blocks() as demo:
        gr.Markdown("## 🤖 Hugging Face AI Agent")
        user_input = gr.Textbox(label="Enter your query", placeholder="e.g. add 5 and 7")
        output = gr.Textbox(label="Agent Response")
        submit_btn = gr.Button("Submit")

        submit_btn.click(fn=ai_agent, inputs=user_input, outputs=output)

    demo.launch()

# =========================================
# RUN AGENT
# =========================================

if __name__ == "__main__":
    print("\n🤖 Hugging Face AI Agent with Gradio UI Started")
    launch_gradio()


Loading Hugging Face model...


Device set to use cpu


Model loaded successfully!

🤖 Hugging Face AI Agent with Gradio UI Started
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ac39316a1e7746189.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
